In [ ]:
# SET UP AND INITIALIZE TOOLS

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
from tqdm import trange

torch.manual_seed(42)

#DEVICE = "cpu"
#DEVICE = "cuda"
DEVICE = "mps"

#DATA = open("shakespeare.txt", "r").read()
DATA = open("kobzar.txt", "r", encoding="utf-8").read()
VOCAB = sorted(set(DATA))
VOCAB_SIZE = len(VOCAB)
VOCAB_MAP = {c:i for i,c in enumerate(VOCAB)}

def encode(s):
    return torch.tensor([VOCAB_MAP[c] for c in s], device=DEVICE)

def decode(l):
    return ''.join([VOCAB[i] for i in l])


DATASET_TRAINING = encode(DATA[:int(len(DATA)*0.9)]).to(DEVICE)
DATASET_TESTING = encode(DATA[int(len(DATA)*0.9):]).to(DEVICE)

def get_batch(input_set, batch_size, block_size):
    ix = torch.randint(0, len(input_set) - block_size, (batch_size, 1), device = input_set.device)
    ix = ix + torch.arange(block_size+1, device = input_set.device)
    batch = input_set[ix]
    return batch[...,:-1], batch[...,1:]

@torch.no_grad()
def estimate_losses(model, iteration=None, save_samples_to_file=None):
    model.eval()
    with torch.autocast('cuda', dtype=torch.bfloat16):
        losses = [ 
            torch.stack([
                model(*get_batch(dataset, 32, model.context_window)) for _ in range(20)
            ]).mean() 
            for dataset in [DATASET_TESTING, DATASET_TRAINING]]

    if save_samples_to_file:
        with open(save_samples_to_file, "a", encoding="utf-8") as f:
            text = (f"Iteration: {iteration}\n" if iteration else "")
            text += f"Loss(test): {losses[0]:.4f}  Loss(train): {losses[1]:.4f}"
            text += "\n-----------------SAMPLE---------------------\n"
            text += decode(model.generate(100))
            text += "\n--------------------------------------------\n\n"
            f.write(text)
    model.train()
    return losses[0], losses[1]

#get_batch(DATASET_TESTING, 4,8)


AssertionError: Torch not compiled with CUDA enabled

In [4]:
'''
serious:
    1. Missed attention scale
    2. Missed final layer norm
    3. Missed positional embeddings size
smaller:
    4. qkv...squeeze(0) is unnecessary
    5. generate(..., prompt_encoded=torch.tensor([])) — mutable default; use None instead.
    6. cross_entropy(...).mean() — cross_entropy already means by default; .mean() is redundant.
    7. for tied weights - don't use bias
'''

class MultiHeadAttention(nn.Module):
    def __init__(self, context_window, embedding_dim, head_size, dropout_p):
        super().__init__()
        assert embedding_dim % head_size == 0, "Error embedding_dim is not divisible by head_size"
        self.heads = embedding_dim // head_size
        self.head_size = head_size 
        self.dropout = nn.Dropout(dropout_p)
        self.qkv = nn.Linear(embedding_dim, 3 * embedding_dim, bias = False)
        self.proj = nn.Linear(embedding_dim, embedding_dim)
        self.register_buffer("clear_mask", torch.tril(torch.ones((context_window, context_window))))

        #Prepare RoPE:
        inv_freq = 1.0 / (10000 ** (torch.arange(0, head_size, 2).float() / head_size))
        pos = torch.arange(context_window).float()
        freqs = torch.einsum("i,j->ij", pos, inv_freq)          # [T, A/2]
        emb = torch.cat([freqs, freqs], dim=-1)                 # [T, A]
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])  # [1,1,T,A]
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def apply_rotary_embeddings(self, q, k):
        # q,k: [B, H, W, A]
        W = q.shape[2]
        cos = self.cos_cached[:, :, :W, :]
        sin = self.sin_cached[:, :, :W, :]

        def rotate_half(x):
            x1 = x[..., : x.shape[-1] // 2]
            x2 = x[..., x.shape[-1] // 2 :]
            return torch.cat((-x2, x1), dim=-1)

        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        return q, k

    def forward(self, x):
        # input: [B,W,E]
        B,W,E = x.shape

        H = self.heads 
        A = self.head_size
        qkv = self.qkv(x)  # [B, W, 3E]
        q,k,v = qkv.view(B,W,3,H,A).permute(2,0,3,1,4) # [B,H,W,A]
        q,k = self.apply_rotary_embeddings(q,k)

        #att = q @ k.transpose(-1,-2)  #[B,H,W,W]
        #att = att * (A**-0.5) # scale
        #att = att.masked_fill(self.clear_mask[:W,:W] == 0.0, float("-inf"))
        #att = F.softmax(att, dim=-1)
        #att = self.dropout(att)
        #out = (att @ v) #[B,H,W,A]

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=(self.dropout.p if self.training else 0.0))

        return self.proj(out.transpose(-3,-2).contiguous().view(B,W,E))

class FeedForward(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.lin = nn.Linear(embedding_dim, 4 * embedding_dim)
        self.gelu = nn.GELU()
        self.proj = nn.Linear(embedding_dim * 4, embedding_dim)
        
    def forward(self, x):  
        x = self.lin(x)
        x = self.gelu(x)
        x = self.proj(x)
        return x

class AttentionBlock(nn.Module):
    def __init__(self, context_window, embedding_dim, head_size, dropout_p):
        super().__init__()
        self.dropout = nn.Dropout(dropout_p)
        self.lnorm1 = nn.LayerNorm(embedding_dim)
        self.lnorm2 = nn.LayerNorm(embedding_dim)
        self.mha = MultiHeadAttention(context_window, embedding_dim, head_size, dropout_p)
        self.ff = FeedForward(embedding_dim)

    def forward(self, x):
        x = x + self.dropout(self.mha(self.lnorm1(x)))
        x = x + self.dropout(self.ff(self.lnorm2(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, vocabulary_size, context_window, embedding_dim, attention_blocks, head_size, dropout_p):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.context_window = context_window
        self.embedding_token = nn.Embedding(vocabulary_size, embedding_dim)
        #self.embedding_posit = nn.Embedding(context_window, embedding_dim)
        self.att = nn.Sequential(*[AttentionBlock(context_window, embedding_dim, head_size, dropout_p) for _ in range(attention_blocks)])
        self.dropout = nn.Dropout(dropout_p)
        self.lnorm = nn.LayerNorm(embedding_dim)
        self.lmh = nn.Linear(embedding_dim, vocabulary_size, bias=False)
        self.lmh.weight = self.embedding_token.weight
        self.register_buffer("positions", torch.arange(0, context_window))

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith('proj.weight'):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * attention_blocks))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, x, y = None):
        x = self.embedding_token(x)# + self.embedding_posit(self.positions[0:x.shape[1]])
        x = self.dropout(x)
        x = self.att(x)
        x = self.lnorm(x)
        x = self.lmh(x)
        if y is not None:
            lg = x.view(-1,self.vocabulary_size)
            return F.cross_entropy(lg, y.contiguous().view(-1))
        else:
            return x

    @torch.no_grad()
    def generate(self, num_of_chars, prompt_encoded = None):
        was_training = self.training
        self.eval()
        out = [0] if prompt_encoded is None else prompt_encoded.tolist()
        for _ in range(num_of_chars):
            inp = torch.tensor(out[-self.context_window:], device=self.lmh.weight.device).view(1,-1)
            l = self(inp)
            r = F.softmax(l[:, -1, :], dim=-1)
            out.append(torch.multinomial(r, num_samples=1).item())
        self.train(was_training)
        return out[(1 if prompt_encoded is None else 0):]
           

m = Transformer(
        VOCAB_SIZE, 
        context_window=128, 
        embedding_dim=64*6, 
        head_size=64, 
        attention_blocks=6, 
        dropout_p=0.1
    ).to(DATASET_TRAINING.device)
#m = torch.compile(m)

def training_loop(model, iterations, learning_rates=[1e-3, 1e-4]):
    best = float('inf')
    decay = [p for n,p in model.named_parameters() if p.dim() >= 2]
    nodecay = [p for n,p in model.named_parameters() if p.dim() < 2]
    o = optim.AdamW([{'params': decay, 'weight_decay': 0.1},
                    {'params': nodecay, 'weight_decay': 0.0}],
                    lr=learning_rates[0], betas=(0.9, 0.95), fused=True)
    sc = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=iterations, eta_min=learning_rates[1])

    print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters.")
    pbar = trange(iterations, desc="Training")
    for i in pbar:
        o.zero_grad()
        
        # CUDA precision
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss = model(*get_batch(DATASET_TRAINING, 32, model.context_window))

        loss.backward()          
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        o.step()
        sc.step()
        if i % 200 == 199:
            ltest, ltrain = estimate_losses(model, iteration=i, save_samples_to_file="samples.txt")
            if ltest < best:
                best = ltest
                torch.save({'model': model.state_dict(), 'iter': i, 'val': ltest}, 'best.pt')
            pbar.set_postfix({"test_loss": f"{ltest:.4f}", "tr_loss": f"{ltrain:.4f}",
                            "best": f"{best:.4f}", "lr": f"{sc.get_last_lr()[0]:.2e}"})
    losses = estimate_losses(model, iteration=i, save_samples_to_file="samples.txt")
    return f"Test loss = {losses[0]:.4f}, Train loss = Test loss = {losses[1]:.4f}"

training_loop(m, 5000, learning_rates=[3e-4, 1e-5])

print(estimate_losses(m))   
print(decode(m.generate(1000)))

#MultiHeadAttention(8, 4, 0.1)(torch.randn((4,10,8)))





#self.embeddings_token = nn.Embedding(vocabulary_size, embedding_dim)
#self.embeddings_pos = nn.Embedding(context_window, embedding_dim)        


NameError: name 'DATASET_TRAINING' is not defined

In [1]:
m.generate(1000, "Вишня")

NameError: name 'm' is not defined

In [ ]:
ckpt = torch.load('best.pt', map_location='cpu')     # weights_only=True is the default now

m = Transformer(VOCAB_SIZE, context_window=128, embedding_dim=64*6,
                head_size=64, attention_blocks=6, dropout_p=0.1)
m.load_state_dict(ckpt['model'])
m.to(DATASET_TRAINING.device)
m.eval()
print(f"loaded iter {ckpt['iter']}, val {ckpt['val']:.4f}")

print(estimate_losses(m)) 

open("generates.txt", "w", encoding="utf-8").write(decode(m.generate(10000)))

loaded iter 3799, val 1.6598
(tensor(1.6605, device='cuda:0'), tensor(0.8550, device='cuda:0'))


UnicodeEncodeError: 'charmap' codec can't encode characters in position 0-4: character maps to <undefined>